In [1]:
!pip install numpy pandas scikit-learn matplotlib seaborn
!pip install keras shap lime streamlit
import torch







In [4]:
import pandas as pd


In [ ]:
df= pd.read_csv(r"C:\Users\abhin\Downloads\archive\Indian_Weather_Dataset.csv")

In [ ]:
df

In [ ]:
print(df.columns)

In [ ]:
X= df[['datetime', 'state', 'city', 'lat', 'lon', 'crops', 'temperature_C',
       'humidity_pct', 'pressure_hPa', 'dew_point_C', 'pressure_trend',
       'solar_radiation_Wm2', 'wind_speed_ms', 'cloud_cover_pct', 'hour',
       'month', 'rain_label', 'wind_direction_deg', 'wind_dir_sin',
       'wind_dir_cos', 'cape', 'et0_mm', 'precip_mm']]

In [ ]:
from sklearn.ensemble import IsolationForest
iso = IsolationForest(
    n_estimators=500,
    contamination=0.01,
    max_samples="auto",  # sample per tree
    random_state=42,
    n_jobs=-1
)


In [ ]:
from sklearn.preprocessing import StandardScaler
# select relevent features
features = [
    "temperature_C",
    "humidity_pct",
    "pressure_hPa",
    "dew_point_C",
    "pressure_trend",
    "solar_radiation_Wm2",
    "wind_speed_ms",
    "cloud_cover_pct",
    "hour",
    "month"
]

X = df[features].dropna().astype("float32")

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
iso.fit(X_scaled)
df["anomaly_iforest"] = iso.predict(X_scaled)  # -1 = anomaly, 1 = normal
df["score_iforest"] = iso.decision_function(X_scaled)  # anomaly score


In [ ]:
print(df["anomaly_iforest"].value_counts())

import matplotlib.pyplot as plt

plt.scatter(df["temperature"], df["pressure"], 
            c=df["anomaly_iforest"], cmap="coolwarm", s=2)
plt.xlabel("Temperature (°C)")
plt.ylabel("Pressure (hPa)")
plt.title("Isolation Forest Anomaly Detection")
plt.show()


In [1]:
import os
import torch

# Use all CPU threads available
num_cores = os.cpu_count()
torch.set_num_threads(num_cores)
torch.set_num_interop_threads(max(1, num_cores // 2))

print("CPU cores:", num_cores)
print("PyTorch threads:", torch.get_num_threads())

CPU cores: 14
PyTorch threads: 14


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler

# -----------------------------
# Step 1 – Define Autoencoder
# -----------------------------
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# -----------------------------
# Step 2 – Prepare Model
# -----------------------------
features = [
    "temperature_C","humidity_pct","pressure_hPa","dew_point_C",
    "pressure_trend","solar_radiation_Wm2","wind_speed_ms",
    "cloud_cover_pct","hour","month"
]

model = Autoencoder(input_dim=len(features))
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
# step 3: Custom Dataset
class WeatherDataset(Dataset):
    def __init__(self, data, scaler=None, fit=False):
        self.scaler = scaler
        X = data[features].dropna().values
        if fit:
            self.X = self.scaler.fit_transform(X)
        else:
            self.X = self.scaler.transform(X)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32)

# -----------------------------
# Step 4 – Stream Data in Chunks
# -----------------------------
chunksize = 5_000_000
reader = pd.read_csv(r"C:\Users\abhin\Downloads\archive\Indian_Weather_Dataset.csv", chunksize=chunksize)

scaler = StandardScaler()
first_chunk = True
all_anomaly_flags = []

for i, chunk in enumerate(reader):
    print(f"Processing chunk {i+1}...")

    # Select features
    X = chunk[features].dropna().values

    # Fit scaler only on first chunk, then reuse

    # Convert to tensor
    dataset = WeatherDataset(chunk, scaler, fit=first_chunk)
    first_chunk = False
    loader = DataLoader(dataset, batch_size=20000, shuffle=True, num_workers=0)

    # Train for a few epochs per chunk
    for epoch in range(1):  # fewer epochs per chunk
        for batch in loader:
            xb = batch[0]
            optimizer.zero_grad()
            recon = model(xb)
            loss = criterion(recon, xb)
            loss.backward()
            optimizer.step()

    # Compute reconstruction error for this chunk
    errors = []
    with torch.no_grad():
        score_loader = DataLoader(dataset, batch_size=50000, num_workers=0)
        for xb in score_loader:
            recon = model(xb)
            batch_errors = torch.mean((recon - xb)**2, dim=1).numpy()
            errors.extend(batch_errors)

    # Flag anomalies (top 1% error in this chunk)
    threshold = np.quantile(errors, 0.99)
    flags = (errors > threshold).astype(int)
    all_anomaly_flags.extend(flags)

# -----------------------------
# Step 4 – Combine Results
# -----------------------------
print("Total anomalies detected:", sum(all_anomaly_flags))




Processing chunk 1...
Processing chunk 2...


In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler

# Configure OpenMP thread affinity before initializing PyTorch operations
# Limits execution to physical P-cores to eliminate E-core barrier stalls
NUM_PHYSICAL_CORES = 6
os.environ["OMP_NUM_THREADS"] = str(NUM_PHYSICAL_CORES)
os.environ["KMP_AFFINITY"] = "granularity=fine,compact,1,0"
os.environ["KMP_BLOCKTIME"] = "1"

torch.set_num_threads(NUM_PHYSICAL_CORES)

# Define Autoencoder Architecture
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# Optimized Dataset class using pre-allocated contiguous float32 memory
class OptimizedWeatherDataset(Dataset):
    def __init__(self, X_data):
        self.X = torch.from_numpy(X_data).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]

# Feature definitions
features = [
    "temperature_C", "humidity_pct", "pressure_hPa", "dew_point_C",
    "pressure_trend", "solar_radiation_Wm2", "wind_speed_ms",
    "cloud_cover_pct", "hour", "month"
]

# Hardware Target Selection: Select Intel Arc GPU ("xpu") if available
device = torch.device("xpu" if hasattr(torch, "xpu") and torch.xpu.is_available() else "cpu")
print("Device:", device)

if device.type == "xpu":
    print("XPU device:", torch.xpu.get_device_name(0))

model = Autoencoder(input_dim=len(features)).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Reduced chunksize prevents system RAM saturation and paging overhead
chunksize = 2_000_000
scaler = StandardScaler()
first_chunk = True
all_anomaly_flags = []

# Accelerated multi-threaded CSV reading via PyArrow engine
reader = pd.read_csv(
    r"C:\Users\abhin\Downloads\archive\Indian_Weather_Dataset.csv",
    chunksize=chunksize,
    usecols=features
)

for i, chunk in enumerate(reader):
    print(f"Processing chunk {i+1}...")
    # Extract feature matrix directly into contiguous float32 array
    X_mat = chunk[features].dropna().to_numpy(dtype=np.float32)

    if first_chunk:
        X_scaled = scaler.fit_transform(X_mat)
        first_chunk = False
    else:
        X_scaled = scaler.transform(X_mat)

    dataset = OptimizedWeatherDataset(X_scaled)

    # Multi-worker DataLoader with pinned memory for non-blocking transfers
    loader = DataLoader(
        dataset,
        batch_size=32768,
        shuffle=True,
        num_workers=0,
        pin_memory=(device.type == "xpu"),
        drop_last=False
    )

    # Corrected Training Loop: Pass full mini-batch tensors
    model.train()
    for epoch in range(1):
        for xb in loader:
            xb = xb.to(device, non_blocking=True)  # Shape: [32768, 10]
            optimizer.zero_grad()
            recon = model(xb)
            loss = criterion(recon, xb)
            loss.backward()
            optimizer.step()

    # Optimized Evaluation Loop
    model.eval()
    errors = []
    score_loader = DataLoader(
        dataset,
        batch_size=65536,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "xpu")
    )

    with torch.no_grad():
        for xb in score_loader:
            xb = xb.to(device, non_blocking=True)
            recon = model(xb)
            batch_errors = torch.mean((recon - xb) ** 2, dim=1).cpu().numpy()
            errors.extend(batch_errors)

    threshold = np.quantile(errors, 0.99)
    flags = (errors > threshold).astype(int)
    all_anomaly_flags.extend(flags)

print("Total anomalies detected:", sum(all_anomaly_flags))

Device: xpu
XPU device: Intel(R) Arc(TM) 130T GPU (8GB)
Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
Processing chunk 6...
Processing chunk 7...
Processing chunk 8...
Processing chunk 9...
Processing chunk 10...
Processing chunk 11...
Processing chunk 12...
Processing chunk 13...
Processing chunk 14...
Processing chunk 15...
Processing chunk 16...
Processing chunk 17...
Processing chunk 18...
Processing chunk 19...
Processing chunk 20...
Processing chunk 21...
Processing chunk 22...
Processing chunk 23...
Processing chunk 24...
Total anomalies detected: 460822


In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler

# Configure OpenMP thread affinity before initializing PyTorch operations
# Limits execution to physical P-cores to eliminate E-core barrier stalls
NUM_PHYSICAL_CORES = 6
os.environ["OMP_NUM_THREADS"] = str(NUM_PHYSICAL_CORES)
os.environ["KMP_AFFINITY"] = "granularity=fine,compact,1,0"
os.environ["KMP_BLOCKTIME"] = "1"

torch.set_num_threads(NUM_PHYSICAL_CORES)

# Define Autoencoder Architecture
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# Optimized Dataset class using pre-allocated contiguous float32 memory
class OptimizedWeatherDataset(Dataset):
    def __init__(self, X_data):
        self.X = torch.from_numpy(X_data).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]

# Feature definitions
features = [
    "temperature_C", "humidity_pct", "pressure_hPa", "dew_point_C",
    "pressure_trend", "solar_radiation_Wm2", "wind_speed_ms",
    "cloud_cover_pct", "hour", "month"
]

# Hardware Target Selection: Select Intel Arc GPU ("xpu") if available
device = torch.device("xpu" if hasattr(torch, "xpu") and torch.xpu.is_available() else "cpu")
print("Device:", device)

if device.type == "xpu":
    print("XPU device:", torch.xpu.get_device_name(0))

model = Autoencoder(input_dim=len(features)).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Reduced chunksize prevents system RAM saturation and paging overhead
chunksize = 2_000_000
scaler = StandardScaler()
first_chunk = True
all_anomaly_flags = []

# Accelerated multi-threaded CSV reading via PyArrow engine
reader = pd.read_csv(
    r"C:\Users\abhin\Downloads\archive\Indian_Weather_Dataset.csv",
    chunksize=chunksize,
    usecols=features
)

for i, chunk in enumerate(reader):
    print(f"Processing chunk {i+1}...")
    # Extract feature matrix directly into contiguous float32 array
    X_mat = chunk[features].dropna().to_numpy(dtype=np.float32)

    if first_chunk:
        X_scaled = scaler.fit_transform(X_mat)
        first_chunk = False
    else:
        X_scaled = scaler.transform(X_mat)

    dataset = OptimizedWeatherDataset(X_scaled)

    # Multi-worker DataLoader with pinned memory for non-blocking transfers
    loader = DataLoader(
        dataset,
        batch_size=32768,
        shuffle=True,
        num_workers=0,
        pin_memory=(device.type == "xpu"),
        drop_last=False
    )

    # Corrected Training Loop: Pass full mini-batch tensors
    model.train()
    for epoch in range(1):
        for xb in loader:
            xb = xb.to(device, non_blocking=True)  # Shape: [32768, 10]
            optimizer.zero_grad()
            recon = model(xb)
            loss = criterion(recon, xb)
            loss.backward()
            optimizer.step()

    # Optimized Evaluation Loop
    model.eval()
    errors = []
    score_loader = DataLoader(
        dataset,
        batch_size=65536,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "xpu")
    )

    with torch.no_grad():
        for xb in score_loader:
            xb = xb.to(device, non_blocking=True)
            recon = model(xb)
            batch_errors = torch.mean((recon - xb) ** 2, dim=1).cpu().numpy()
            errors.extend(batch_errors)

    threshold = np.quantile(errors, 0.99)
    flags = (errors > threshold).astype(int)
    all_anomaly_flags.extend(flags)

print("Total anomalies detected:", sum(all_anomaly_flags))

Device: xpu
XPU device: Intel(R) Arc(TM) 130T GPU (8GB)
Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
Processing chunk 6...
Processing chunk 7...
Processing chunk 8...
Processing chunk 9...
Processing chunk 10...
Processing chunk 11...
Processing chunk 12...
Processing chunk 13...
Processing chunk 14...
Processing chunk 15...
Processing chunk 16...
Processing chunk 17...
Processing chunk 18...
Processing chunk 19...
Processing chunk 20...
Processing chunk 21...
Processing chunk 22...
Processing chunk 23...
Processing chunk 24...
Total anomalies detected: 460822


In [1]:
python -m pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/xpu


SyntaxError: invalid syntax (1454972631.py, line 1)